# AWS-Native Python ML Pipeline Workbench

This notebook demonstrates the recommended boundary:

```text
Terraform / CDK / CloudFormation
    provisions durable AWS resources
                    ↓
Infrastructure readiness gate
    verifies resources and permissions
                    ↓
Jupyter + Python
    explores data and submits jobs
                    ↓
SageMaker Processing / Training
    executes heavy Python workloads
                    ↓
S3 / Model Registry / Endpoint
    stores and serves managed artifacts
```

The notebook deliberately **does not create production infrastructure by default**.

It consumes Terraform outputs, verifies that AWS is ready, and only then allows Processing or Training jobs to be submitted.

## 1. Why perform infrastructure readiness checks?

Yes, this is a good practice.

The notebook should fail before expensive or long-running work when:

- AWS credentials are missing.
- The configured account or region is wrong.
- The S3 artifact bucket does not exist.
- The SageMaker execution role does not exist.
- The caller cannot access SageMaker APIs.
- Required configuration values are absent.

This is a **preflight gate**, not infrastructure provisioning.

Production IaC should normally run through CI/CD or a controlled operator workflow:

```bash
terraform init
terraform plan
terraform apply
./scripts/export_terraform_outputs.sh
```

The notebook begins after that workflow.

In [ ]:
from __future__ import annotations

from io import StringIO
from pathlib import Path
import json
import os
import sys

import boto3
import botocore
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()

# Allow execution when launched from notebooks/.
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

CONFIG_PATH = REPO_ROOT / "config" / "aws_config.json"
EXAMPLE_CONFIG_PATH = REPO_ROOT / "config" / "aws_config.example.json"

print("Repository root:", REPO_ROOT)
print("Expected configuration:", CONFIG_PATH)

## 2. Infrastructure provisioning boundary

The notebook does not automatically run `terraform apply`.

For demonstration, these are the commands that should run **before** this notebook:

```bash
cd infra/terraform
terraform init
terraform plan -out=tfplan
terraform apply tfplan

cd ../..
./scripts/export_terraform_outputs.sh
```

A GitLab runner can execute those commands with protected credentials and manual approval.

The notebook can display or validate IaC, but allowing an interactive notebook to become the source of truth for production infrastructure would create ordering, state, review, and reproducibility problems.

In [ ]:
# Optional inspection only: show whether Terraform is installed.
# This does not provision anything.

import shutil

terraform_path = shutil.which("terraform")
aws_cli_path = shutil.which("aws")

print("Terraform:", terraform_path or "not installed in this environment")
print("AWS CLI:", aws_cli_path or "not installed in this environment")

## 3. Load configuration exported from Terraform

If the real configuration file does not exist, this notebook loads the example so you can still walk through every section without making AWS calls.

In [ ]:
def load_config() -> tuple[dict, bool]:
    if CONFIG_PATH.exists():
        return json.loads(CONFIG_PATH.read_text(encoding="utf-8")), True

    print(
        "Real AWS configuration is not present. "
        "Loading the example in walkthrough mode."
    )
    return json.loads(EXAMPLE_CONFIG_PATH.read_text(encoding="utf-8")), False


config, LIVE_AWS_MODE = load_config()

config

## 4. Infrastructure readiness gate

This check confirms that IaC-created resources are ready before the notebook submits SageMaker jobs.

It verifies:

1. AWS identity
2. Selected region
3. S3 artifact bucket
4. SageMaker execution role
5. SageMaker API availability

It does not prove every future action is authorized, because IAM can include conditions and service-specific constraints. It is still valuable as an early failure gate.

In [ ]:
from botocore.exceptions import ClientError, NoCredentialsError


def check_aws_infrastructure(config: dict) -> dict:
    required = [
        "aws_region",
        "artifact_bucket",
        "sagemaker_execution_role_arn",
        "model_package_group_name",
    ]

    missing = [
        key
        for key in required
        if not config.get(key)
        or str(config[key]).startswith("replace-with")
    ]

    if missing:
        raise ValueError(
            "Configuration is incomplete. Missing real values for: "
            + ", ".join(missing)
        )

    session = boto3.Session(
        region_name=config["aws_region"]
    )

    sts = session.client("sts")
    s3 = session.client("s3")
    iam = session.client("iam")
    sagemaker = session.client("sagemaker")

    identity = sts.get_caller_identity()

    s3.head_bucket(
        Bucket=config["artifact_bucket"]
    )

    role_name = (
        config["sagemaker_execution_role_arn"]
        .rsplit("/", 1)[-1]
    )

    iam.get_role(RoleName=role_name)

    # A lightweight API call verifies endpoint reachability and
    # at least one SageMaker permission.
    sagemaker.list_training_jobs(MaxResults=1)

    return {
        "account_id": identity["Account"],
        "caller_arn": identity["Arn"],
        "region": config["aws_region"],
        "artifact_bucket": config["artifact_bucket"],
        "sagemaker_role": config[
            "sagemaker_execution_role_arn"
        ],
        "status": "READY",
    }


if LIVE_AWS_MODE:
    try:
        readiness = check_aws_infrastructure(config)
        display(pd.DataFrame([readiness]))
        INFRASTRUCTURE_READY = True
    except (
        ClientError,
        NoCredentialsError,
        ValueError,
    ) as exc:
        INFRASTRUCTURE_READY = False
        print("Infrastructure readiness failed:")
        print(exc)
else:
    INFRASTRUCTURE_READY = False
    print(
        "Walkthrough mode: readiness calls are skipped. "
        "Provision infrastructure and export outputs to enable live mode."
    )

## 5. Optional IaC call location

This is where an operator could invoke IaC from a notebook, but it is intentionally commented out:

```python
# !terraform -chdir={REPO_ROOT / "infra/terraform"} init
# !terraform -chdir={REPO_ROOT / "infra/terraform"} plan -out=tfplan
# !terraform -chdir={REPO_ROOT / "infra/terraform"} apply tfplan
# !bash {REPO_ROOT / "scripts/export_terraform_outputs.sh"}
```

That can be useful for a disposable training sandbox. For shared or production infrastructure, run it from a protected CI/CD job with review and remote Terraform state.

## 6. Bring your own small dataset or use defaults

Keyboard data is capped at 1,000 characters. The default data is designed for a churn-classification demonstration.

In [ ]:
from sklearn.datasets import make_classification

MAX_KEYBOARD_CHARACTERS = 1_000


def create_default_data(rows: int = 1_000) -> pd.DataFrame:
    X, y = make_classification(
        n_samples=rows,
        n_features=6,
        n_informative=4,
        n_redundant=1,
        weights=[0.7, 0.3],
        random_state=42,
    )

    rng = np.random.default_rng(42)

    df = pd.DataFrame(
        X,
        columns=[
            "usage_score",
            "support_calls",
            "billing_variance",
            "engagement_score",
            "tenure_signal",
            "risk_signal",
        ],
    )

    df["customer_id"] = [
        f"CUST-{i:05d}" for i in range(rows)
    ]
    df["region"] = rng.choice(
        ["North", "South", "East", "West"],
        rows,
    )
    df["plan_type"] = rng.choice(
        ["Basic", "Standard", "Premium"],
        rows,
    )
    df["monthly_spend"] = (
        60 + 10 * df["usage_score"]
        + rng.normal(0, 12, rows)
    ).clip(5, 250)
    df["signup_date"] = (
        pd.Timestamp("2022-01-01")
        + pd.to_timedelta(
            rng.integers(0, 1_200, rows),
            unit="D",
        )
    )
    df["churned"] = y
    return df


def choose_data() -> pd.DataFrame:
    print("1 or blank: default data")
    print("2: pasted CSV")
    print("3: pasted JSON")

    choice = input("Choice [1]: ").strip()

    if choice in {"", "1"}:
        return create_default_data()

    text = input(
        f"Paste up to {MAX_KEYBOARD_CHARACTERS} characters: "
    ).strip()

    if not text:
        return create_default_data()

    if len(text) > MAX_KEYBOARD_CHARACTERS:
        raise ValueError("Keyboard input is too long.")

    if choice == "2":
        return pd.read_csv(StringIO(text))

    if choice == "3":
        value = json.loads(text)
        return pd.DataFrame(value)

    return create_default_data()


data = choose_data()
display(data.head())
print("Shape:", data.shape)

## 7. Local exploration before cloud execution

It is reasonable to inspect a sample with pandas locally before submitting cloud jobs.

The notebook should not process the full production dataset merely because it can. Large processing belongs in SageMaker Processing, Glue, EMR, Ray, or another managed compute environment.

In [ ]:
display(
    pd.DataFrame({
        "dtype": data.dtypes.astype(str),
        "missing": data.isna().sum(),
        "unique": data.nunique(dropna=False),
    })
)

data.select_dtypes(include="number").hist(
    figsize=(12, 8),
    bins=25,
)
plt.tight_layout()
plt.show()

## 8. Upload source data to S3

The upload cell is gated by infrastructure readiness.

```text
Notebook memory
    ↓ boto3 upload
S3 raw prefix
    ↓ SageMaker Processing input
Temporary processing container
```

In [ ]:
local_data_path = REPO_ROOT / "data" / "customer_churn.csv"
local_data_path.parent.mkdir(exist_ok=True)
data.to_csv(local_data_path, index=False)

raw_s3_uri = (
    f"s3://{config['artifact_bucket']}/"
    f"{config['raw_prefix']}/customer_churn.csv"
)

if INFRASTRUCTURE_READY:
    s3 = boto3.client(
        "s3",
        region_name=config["aws_region"],
    )
    s3.upload_file(
        str(local_data_path),
        config["artifact_bucket"],
        f"{config['raw_prefix']}/customer_churn.csv",
    )
    print("Uploaded:", raw_s3_uri)
else:
    print("Dry run; would upload to:", raw_s3_uri)

## 9. Submit SageMaker Processing

The notebook submits `src/preprocess.py`; the heavy transformation runs on temporary SageMaker-managed compute.

The notebook is acting as a control plane, not as the data-processing machine.

In [ ]:
processed_s3_uri = (
    f"s3://{config['artifact_bucket']}/"
    f"{config['processed_prefix']}/"
)

if INFRASTRUCTURE_READY:
    from sagemaker.processing import (
        ProcessingInput,
        ProcessingOutput,
        SKLearnProcessor,
    )

    processor = SKLearnProcessor(
        framework_version="1.2-1",
        role=config["sagemaker_execution_role_arn"],
        instance_type="ml.m5.xlarge",
        instance_count=1,
        base_job_name="aws-native-python-preprocess",
    )

    processor.run(
        code=str(REPO_ROOT / "src" / "preprocess.py"),
        arguments=[
            "--input",
            "/opt/ml/processing/input/customer_churn.csv",
            "--output-dir",
            "/opt/ml/processing/output",
        ],
        inputs=[
            ProcessingInput(
                source=raw_s3_uri,
                destination="/opt/ml/processing/input",
            )
        ],
        outputs=[
            ProcessingOutput(
                source="/opt/ml/processing/output",
                destination=processed_s3_uri,
            )
        ],
        wait=True,
        logs=True,
    )
else:
    print("Dry run Processing output:", processed_s3_uri)
    print("Script:", REPO_ROOT / "src" / "preprocess.py")

## 10. Submit SageMaker Training

The training script remains normal Python. SageMaker provides:

- Temporary compute
- Container runtime
- S3 input channels
- Logs
- Model artifact packaging
- Automatic shutdown

In [ ]:
if INFRASTRUCTURE_READY:
    from sagemaker.inputs import TrainingInput
    from sagemaker.sklearn.estimator import SKLearn

    estimator = SKLearn(
        entry_point="train.py",
        source_dir=str(REPO_ROOT / "src"),
        role=config["sagemaker_execution_role_arn"],
        instance_type="ml.m5.xlarge",
        instance_count=1,
        framework_version="1.2-1",
        py_version="py3",
        base_job_name="aws-native-python-train",
        hyperparameters={
            "target": "churned",
            "n-estimators": 300,
            "max-depth": 12,
        },
    )

    estimator.fit({
        "train": TrainingInput(
            s3_data=processed_s3_uri,
            content_type="application/x-parquet",
        )
    })

    print("Model artifact:", estimator.model_data)
else:
    print("Dry run: training would read from", processed_s3_uri)

## 11. Evaluation, quality gate, and registration

Best practice is to evaluate on separate managed compute, write a metrics artifact to S3, and register only when thresholds pass.

Conceptual condition:

```python
if evaluation["roc_auc"] >= 0.88:
    register_model()
else:
    stop_pipeline()
```

In a durable workflow, this becomes a SageMaker Pipeline `ConditionStep`, not an interactive notebook `if` statement.

In [ ]:
MINIMUM_ROC_AUC = 0.88

# Placeholder for the output of the managed evaluation step.
example_evaluation = {
    "roc_auc": 0.91,
    "minimum_required": MINIMUM_ROC_AUC,
}

example_evaluation["approved_for_registration"] = (
    example_evaluation["roc_auc"]
    >= example_evaluation["minimum_required"]
)

example_evaluation

## 12. Optional endpoint deployment

Deployment remains disabled by default because a persistent endpoint can continue incurring charges after the notebook closes.

For learning:

```python
# predictor = estimator.deploy(
#     initial_instance_count=1,
#     instance_type="ml.m5.large",
#     endpoint_name="aws-native-python-ml-demo",
# )
```

For production, deploy an approved Model Registry version through a controlled promotion workflow.

## 13. Where GitLab CI fits

A GitLab runner should perform control-plane work:

```text
lint and unit tests
    ↓
Terraform validation and plan
    ↓
protected Terraform apply
    ↓
publish configuration outputs
    ↓
build or validate ML containers
    ↓
upsert SageMaker Pipeline definition
    ↓
start or promote pipeline executions
```

It should normally **not** load the full training dataset into the runner or train large models on runner compute.

## 14. Final architecture

```text
Git repository
├── Terraform: durable AWS foundation
├── Python: data/model logic
├── Notebook: exploration and controlled job submission
└── GitLab CI: validation and promotion
                │
                ▼
AWS account
├── S3: raw, processed, evaluation, model artifacts
├── IAM: execution roles and least-privilege access
├── SageMaker Processing: temporary preprocessing compute
├── SageMaker Training: temporary training compute
├── SageMaker Pipelines: durable ML DAG
├── Model Registry: versions and approvals
└── Endpoint/Batch Transform: managed inference
```

The readiness gate is the bridge between IaC and the notebook:

```text
IaC reports successful deployment
        ↓
Terraform outputs exported
        ↓
Notebook verifies real AWS resources
        ↓
Only then submit Processing and Training
```

## 15. Reusable AWS SDK logic

The notebook should not contain every low-level API call inline. It imports the same `AwsMlJobs` class used by the deployable EKS controller.

This gives three execution modes:

```text
Notebook → AwsMlJobs → SageMaker
CLI/script → AwsMlJobs → SageMaker
EKS FastAPI → AwsMlJobs → SageMaker
```

The ML control logic therefore remains Python and testable instead of being trapped inside notebook cells.

In [ ]:
from src.aws_jobs import AwsMlConfig, AwsMlJobs

# These image URIs must be supplied by your AWS configuration.
# SageMaker framework images or your own ECR images can be used.
processing_image_uri = config.get(
    "processing_image_uri",
    "REPLACE_WITH_PROCESSING_IMAGE",
)
training_image_uri = config.get(
    "training_image_uri",
    "REPLACE_WITH_TRAINING_IMAGE",
)

if INFRASTRUCTURE_READY and not processing_image_uri.startswith("REPLACE"):
    aws_jobs = AwsMlJobs(
        AwsMlConfig(
            region=config["aws_region"],
            bucket=config["artifact_bucket"],
            role_arn=config["sagemaker_execution_role_arn"],
            processing_image_uri=processing_image_uri,
            training_image_uri=training_image_uri,
        )
    )
    display(pd.DataFrame([aws_jobs.readiness()]))
else:
    print("Walkthrough mode: AwsMlJobs is importable but live calls are disabled.")

## 16. Kubernetes deployment

The deployable component is not the notebook itself. It is a small Python control-plane service packaged with Docker and Helm:

```text
app/main.py
    FastAPI endpoints

src/aws_jobs.py
    boto3 SageMaker API calls

Dockerfile
    portable application image

Helm chart
    Kubernetes Deployment, Service and ServiceAccount

Terraform
    ECR repository and IRSA role
```

The controller can run on EKS while SageMaker continues to execute temporary Processing and Training jobs.

An alternative is the SageMaker ACK controller, which lets Kubernetes custom resources represent SageMaker jobs. This repository uses Python SDK calls because the educational goal is to expose the AWS API logic directly.

In [ ]:
# Deployment commands are intentionally displayed rather than automatically run.

deployment_commands = f'''
docker build -t aws-ml-controller {REPO_ROOT}

# Push to the ECR repository created by Terraform.
docker tag aws-ml-controller:latest $ECR_REPOSITORY:latest
docker push $ECR_REPOSITORY:latest

kubectl create namespace ml-platform --dry-run=client -o yaml | kubectl apply -f -

helm upgrade --install aws-ml-controller \
  {REPO_ROOT / "deploy/helm/aws-ml-controller"} \
  --namespace ml-platform \
  --set image.repository=$ECR_REPOSITORY \
  --set serviceAccount.roleArn=$CONTROLLER_IRSA_ROLE_ARN \
  --set env.artifactBucket={config["artifact_bucket"]} \
  --set env.sagemakerExecutionRoleArn={config["sagemaker_execution_role_arn"]}
'''
print(deployment_commands)